# Update meta data and objects 

In [ ]:
import scvi
import scanpy as sc

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

import os
import sys

## Root directory

In [ ]:
os.chdir('/research/peer/fdeckert/FD20250213HSCT')

## Custom modules

In [ ]:
sys.path.append('bin/')
from adata_qc import *

## Settup rpy2 

In [ ]:
os.environ['R_HOME'] = '/nobackup/peer/fdeckert/miniconda3/envs/r.4.4.1-FD20250213HSCT/lib/R'

In [ ]:
import rpy2
%load_ext rpy2.ipython

## Figures 

In [ ]:
sc.set_figure_params(figsize=(5, 5), dpi_save=1200, fontsize=12, frameon=False, facecolor='white')
mpl.rcParams['figure.facecolor'] = 'white'

# Import 

In [ ]:
adata = sc.read_h5ad('data/object/pp.h5ad')

# SCVI on all CD4+ T cells

In [ ]:
adata_0 = adata[adata.obs.celltype_low.isin(['$T_{h}\ E/EM$', '$T_{h}\ N/CM$', '$T_{h}\ RM$'])]

In [ ]:
adata_0 = adata_0[:, (adata_0.X>=3).sum(axis=0)>=3]

In [ ]:
adata_0 = adata_0.copy()

In [ ]:
cache_scvi = True

In [ ]:
if not cache_scvi: 
    
    scvi.model.SCVI.setup_anndata(
    
        adata_0,
        batch_key='sample_name', 
        continuous_covariate_keys=['S_ucell', 'G2M_ucell'], 
        labels_key='celltype_low'
    
    )
    
    model = scvi.model.SCVI(
    
        adata_0, 
        n_latent=30, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'

    )
    
    max_epochs = int(np.min([round((20000 / adata_0.n_obs) * 400), 400]))
    
    model.train(max_epochs=max_epochs, check_val_every_n_epoch=1)
    
    model.save('data/object/scvi/pseudotime/model_0/', overwrite=True)
    
else: 
    
    model = scvi.model.SCVI.load('data/object/scvi/pseudotime/model_0/', adata=adata_0)

In [ ]:
cache_dim = True

In [ ]:
if not cache_dim: 
    
    adata_0.obsm['latent'] = model.get_latent_representation()

    adata_0 = adata_0.raw.to_adata()
    adata_0.raw = adata_0.copy()
    
    sc.pp.neighbors(adata_0, n_neighbors=15, use_rep='latent')
    sc.tl.umap(adata_0, min_dist=1)
    
    adata_0.write_h5ad('data/object/scvi/pseudotime/adata_0.h5ad')
    
else: 
    
    adata_0 = sc.read_h5ad('data/object/scvi/pseudotime/adata_0.h5ad')

In [ ]:
sc.pl.umap(adata_0, color=['celltype_low', 'celltype_high', 'celltype_domain', 'patient_id', 'hto_demux_class', 'time_point', 'msCC_label', 'genotype_class', 'clone_id_size'], size=50, ncols=5, wspace=0.1, legend_loc='on data')

# SCVI on all CD8+ T cells

In [ ]:
adata_1 = adata[adata.obs.celltype_low.isin(['$T_{c}\ E/EM$', '$T_{c}\ N/CM$', '$T_{c}\ RM$'])]

In [ ]:
adata_1 = adata_1[:, (adata_1.X>=3).sum(axis=0)>=3]

In [ ]:
adata_1 = adata_1.copy()

In [ ]:
cache_scvi = False

In [ ]:
if not cache_scvi: 
    
    scvi.model.SCVI.setup_anndata(
    
        adata_1,
        batch_key='sample_name', 
        continuous_covariate_keys=['S_ucell', 'G2M_ucell'], 
        labels_key='celltype_low'
        
    )
    
    model = scvi.model.SCVI(
    
        adata_1, 
        n_latent=30, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'

    )
    
    max_epochs = int(np.min([round((20000 / adata_1.n_obs) * 400), 400]))
    
    model.train(max_epochs=max_epochs, check_val_every_n_epoch=1)
    
    model.save('data/object/scvi/pseudotime/model_1/', overwrite=True)
    
else: 
    
    model = scvi.model.SCVI.load('data/object/scvi/pseudotime/model_1/', adata=adata_1)

In [ ]:
cache_dim = False

In [ ]:
if not cache_dim: 
    
    adata_1.obsm['latent'] = model.get_latent_representation()

    adata_1 = adata_1.raw.to_adata()
    adata_1.raw = adata_1.copy()
    
    sc.pp.neighbors(adata_1, n_neighbors=15, use_rep='latent')
    sc.tl.umap(adata_1, min_dist=1)
    
    adata_1.write_h5ad('data/object/scvi/pseudotime/adata_1.h5ad')
    
else: 
    
    adata_1 = sc.read_h5ad('data/object/scvi/pseudotime/adata_1.h5ad')

In [ ]:
sc.pl.umap(adata_1, color=['celltype_low', 'celltype_high', 'celltype_domain', 'patient_id', 'hto_demux_class', 'time_point', 'msCC_label', 'genotype_class', 'clone_id_size'], size=50, ncols=5, wspace=0.1, legend_loc='on data')